<a href="https://colab.research.google.com/github/mdudavillela/gradient-convergence-mlp/blob/main/notebooks/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Main Notebook

Este notebook dita o projeto: ele não tem a lógica pesada em si, só chama funções que moram nos módulos dentro de `src/` (mesmo papel que um `main.R` com `source()` cumpriria se fosse programado em R).

**Como funciona esse notebook?**
```
main.ipynb -> clona o repositório -> importa coleta.py -> pega a API key -> roda coletar_dataset() -> gera data/tmdb_raw.csv
```

Por enquanto ele cobre só o "pré-processamento": importar/coletar o dataset. Os próximos módulos entram depois, cada um importado aqui quando chegar a vez.

## 1. Clonar o repositório

O colab roda numa máquina virtual temporária, ela não tem acesso ao repositório do github por padrão, então o primeiro passo é baixar uma cópia dele pra dentro da máquina (`src/coleta.py`).

- `!git clone {REPO_URL}` — o `!` diz ao notebook "isso não é python, é um comando de terminal". É o mesmo `git clone` que seria usado fora do colab.
- `%cd {REPO_DIR}` — o `%` marca um comando "mágico" do Jupyter. Esse em específico muda a pasta de trabalho pra dentro do repositório clonado, pra que os caminhos relativos usados nas próximas células (tipo `"src"` ou `"data/tmdb_raw.csv"`) funcionem certinho.

In [ ]:
import os  # biblioteca pra interagir com o sistema operacional -- aqui, só pra checar se a pasta já existe

REPO_URL = "https://github.com/mdudavillela/gradient-convergence-mlp.git"
REPO_DIR = "gradient-convergence-mlp"  # nome da pasta que o clone cria (sempre o nome do repositório)

# só clona se a pasta ainda não existir -- evita erro caso essa célula rode duas vezes na mesma sessão
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"Pasta '{REPO_DIR}' já existe — pulando o clone.")

%cd {REPO_DIR}

## 2. Importar o módulo de coleta

`sys.path` é a lista de pastas onde o Python procura por módulos quando você escreve `import algo`. Por padrão, `src/` não está nessa lista -- sem esse passo, `from coleta import coletar_dataset` daria `ModuleNotFoundError`.

É o equivalente em espírito ao `source("coleta.R")` do R: lá você aponta pro arquivo direto; aqui, você aponta pra pasta (`sys.path.append`) e importa pelo nome do arquivo, sem a extensão `.py`.

In [ ]:
import sys
sys.path.append("src")

from coleta import coletar_dataset

print("Módulo 'coleta' importado com sucesso.")

## 3. Configurar a API key

Tenta carregar a chave dos **Secrets** do Colab primeiro (configure uma vez: ícone 🔑 na barra lateral → "Add new secret" → nome `TMDB_API_KEY`). Se não achar -- por exemplo, rodando fora do Colab, ou esqueceu de configurar -- cai automaticamente no modo manual (`getpass`, que esconde o que você digita).

In [ ]:
import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("TMDB_API_KEY")
    print("API key carregada dos Secrets do Colab.")
except Exception:
    api_key = getpass.getpass("Cole sua API Key (v3 auth) do TMDB: ")

## 4. Rodar a coleta

Aqui chamamos a função "chefe" do `coleta.py`. O notebook não precisa saber COMO a coleta funciona por dentro -- só precisa saber o nome da função, quais parâmetros ela espera, e o que ela devolve (um `DataFrame`). Essa separação entre "o que fazer" (aqui) e "como fazer" (dentro do módulo) é o ganho principal de organizar o código assim.

Na primeira vez, demora alguns minutos (milhares de requisições à API). Depois disso -- mesmo numa sessão nova do Colab -- se `data/tmdb_raw.csv` já estiver commitado no repositório, essa célula carrega ele na hora em vez de recoletar (é a lógica de cache de dentro da `coletar_dataset`).

**Atenção aos nomes dos parâmetros abaixo** -- precisam bater exatamente com os nomes definidos na sua versão do `coleta.py` (`n_paginas`, `qtd_votos_min`, `numero_threads`, `atualizar_do_zero`), senão dá erro de parâmetro desconhecido, igual aconteceu com `filtros`/`params` antes.

In [ ]:
df_raw = coletar_dataset(
    api_key = api_key,
    cache_path = "data/tmdb_raw.csv",
    n_paginas = 250,          # ~20 filmes por página -> até ~5000 filmes candidatos
    qtd_votos_min = 100,      # mínimo de votos para a nota do público ser confiável
    numero_threads = 10,      # requisições simultâneas na busca de detalhes
    atualizar_do_zero = False, # mude para True só se quiser recoletar dados atualizados
)

print(df_raw.shape)
df_raw.head()

## 5. Próximo passo: levar o CSV para o repositório

Se a coleta acabou de rodar pela primeira vez, `data/tmdb_raw.csv` existe só nesta máquina temporária do Colab -- assim que a sessão desconectar, ele some, porque ainda não está no GitHub.

Jeito mais simples: baixe o arquivo do Colab (ícone de pasta 📁 na lateral → botão direito em `data/tmdb_raw.csv` → Download) e adicione ele ao repositório do mesmo jeito que você já fez com o `coleta.py` (Add file → Create new file ou Upload files → commit).

Depois disso, qualquer sessão futura (sua ou da orientadora, se ela quiser reproduzir) já encontra o cache pronto -- sem precisar de API key nem esperar os minutos de coleta de novo.